# Train the PM2.5 forecasting models

English | [简体中文](model_training.zh-CN.ipynb)

Discover the forecasting classes in the [model package](source/models/README.md)
and train their declared configurations on the chronological training split.
Evaluate each prediction budget on the validation split and save a checkpoint
for each configuration. The [test notebook](model_test_comparison.ipynb)
loads these checkpoints and runs inference on the reserved test split.

## Prepare the input data

Run these commands from the project root when preparing inputs for the first
time. Use the project's [pinned dependencies](requirements.txt) in the kernel.

```bash
python -B -m source.prepare
python -B -m source.prepare_recent_weather
python -B -m source.build_experiment_matrices
```

Open this notebook from the project root. Set a fresh `RUN_DIR` for a new
training run and use the same path in the test notebook.

In [ ]:
import sys

sys.dont_write_bytecode = True

from source.comparison import comparison_configs, train_comparison
from source.experiment_paths import CHECKPOINTS, MATRIX
from source.models import discover_models

TRAIN_SEED = 42
THREADS = 4
RUN_DIR = CHECKPOINTS

models = discover_models()
configs = comparison_configs(TRAIN_SEED)
print("Model packages:", ", ".join(models))
print("Configurations:", len(configs))

## Fit and validate

The legacy East and five-region models use 25,668 training origins. The
additional recipes declare their input profiles and whether to use the
expanded training cohort of 59,601 origins. Compact-feature median imputation
is fitted on the corresponding training cohort and saved with the model.
All configurations use the same 3,331 validation origins.

Budget selection in this notebook uses the minimum validation MSE among each
configuration's declared budgets. The final fit follows each model's capacity
policy. The legacy command-line runner uses chronological cross-validation
for budget selection.

In [ ]:
entries = train_comparison(MATRIX, RUN_DIR, seed=TRAIN_SEED, threads=THREADS)
validation_order = sorted(entries, key=lambda name: entries[name]["validation_mse"])
print("Validation selection:", validation_order[0])